# Scroll ink sensitivity, stage 0: baseline on a known-ink segment

Runs the official Vesuvius Challenge 9 µm ink model (`scrollprize/ink_9um`, MIT) on **PHerc. 139, segment w045**, a segment held out from training,
at **9.362 µm**: the same scan setting as the unread First Letters scrolls. It then compares the result with the published 2.4 µm ink map of the same segment.

Settings → Accelerator → **GPU T4 x2**, Internet **on**. *Run all*, then copy the printed summary at the end.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Official code: ScrollPrize/villa, branch merge-ink-pipelines (ink-detection/koine_machines)
!GIT_LFS_SKIP_SMUDGE=1 git clone -q --depth 1 --branch merge-ink-pipelines https://github.com/ScrollPrize/villa /kaggle/working/villa
!pip install -q uv imagecodecs
%cd /kaggle/working/villa/ink-detection
!uv sync -q 2>&1 | tail -5
!uv run python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
import glob
import os
import subprocess
import time

import numpy as np
from huggingface_hub import hf_hub_download

W = "/kaggle/working"
os.makedirs(f"{W}/out", exist_ok=True)
B = "https://vesuvius-challenge-open-data.s3.amazonaws.com"
SEG = "PHerc0139/segments/20260126000000-w045_2026012619"
VOL = f"{B}/{SEG}/surface-volumes/9.362um-1.2m-113keV-volume-20250728140407.zarr"
# The two default checkpoints used by the First Letters survey
CKPTS = [hf_hub_download("scrollprize/ink_9um", f)
         for f in ("hybrid_3d2d-seed42/step-075000.pth", "hybrid_3d2d-seed43/step-020000.pth")]
print(CKPTS)

In [ ]:
# Official inference, both depth directions, for each checkpoint
times = {}
for i, ck in enumerate(CKPTS):
    out = f"{W}/out/w045_ck{i}.tif"
    t0 = time.time()
    r = subprocess.run(["uv", "run", "python", "-m", "koine_machines.inference.infer",
                        VOL, ck, out, "--direction", "both", "--batch-size", "16"],
                       capture_output=True, text=True)
    times[i] = time.time() - t0
    print(f"checkpoint {i}: exit {r.returncode}, {times[i]:.0f} s")
    print(r.stdout[-1500:])
    print(r.stderr[-2500:])
print(sorted(glob.glob(f"{W}/out/*")))

In [ ]:
# Average every output map, then compare with the published 2.4 um map of the same segment
import urllib.request

from PIL import Image

# Decode the LZW TIFFs inside the official environment (it has imagecodecs), then load as NumPy
subprocess.run(["uv", "run", "python", "-c", "import glob, numpy as np, tifffile; "
                "[np.save(p.rsplit('.', 1)[0] + '.npy', tifffile.imread(p)) for p in sorted(glob.glob('/kaggle/working/out/w045_ck*.tif*'))]"],
               check=True, cwd="/kaggle/working/villa/ink-detection")
maps = sorted(glob.glob(f"{W}/out/w045_ck*.npy"))
arrs = [np.load(p).astype(np.float32) for p in maps]
for p, a in zip(maps, arrs):
    print(os.path.basename(p), a.shape, a.dtype, float(a.min()), float(a.max()), round(float(a.mean()), 2))
ours = np.mean([a if a.ndim == 2 else a.squeeze() for a in arrs], axis=0)

ref_url = (f"{B}/{SEG}/ink-detection/downsampled/PHerc0139-20260126000000-2.399um-0.22m-78keV-volume-"
           "20260102150214-20260417190342-new_canon_autoresearch_recipe-tile256-stride128-ds8.jpg")
urllib.request.urlretrieve(ref_url, f"{W}/out/ref_2p4um_ds8.jpg")
ref = np.asarray(Image.open(f"{W}/out/ref_2p4um_ds8.jpg").convert("L"), dtype=np.float32)
print("ours", ours.shape, "aspect", round(ours.shape[1] / ours.shape[0], 4),
      "| ref", ref.shape, "aspect", round(ref.shape[1] / ref.shape[0], 4))

def resized(a, shape):
    return np.asarray(Image.fromarray(a).resize((shape[1], shape[0]), Image.BILINEAR), dtype=np.float32)

def corr(a, b, m):
    a, b = a[m], b[m]
    return float(np.corrcoef(a, b)[0, 1])

variants = {"as is": ours, "flip up-down": ours[::-1], "flip left-right": ours[:, ::-1], "rotate 180": ours[::-1, ::-1]}
res = {}
for k, v in variants.items():
    o = resized(np.ascontiguousarray(v), ref.shape)
    m = (ref > 0) & (o > 0)
    res[k] = (corr(o, ref, m), o, m)
    print(f"{k:16s} r = {res[k][0]:.3f}")
best = max(res, key=lambda k: res[k][0])
r_best, o_best, m_best = res[best]
# Null: the same map shifted by 1/8 of the width, as the survey's shifted-map control
sh = ref.shape[1] // 8
r_null = corr(np.roll(o_best, sh, axis=1), ref, m_best & np.roll(m_best, sh, axis=1))
print(f"best: {best}, r = {r_best:.3f}; shifted-map null r = {r_null:.3f}")

disp = np.clip((o_best / 255.0 - 0.25) / 0.5, 0, 1)
side = np.concatenate([(disp * 255).astype(np.uint8), ref.astype(np.uint8)], axis=1)
Image.fromarray(side).save(f"{W}/out/w045_9um_vs_published_2p4um.png")
print("saved", f"{W}/out/w045_9um_vs_published_2p4um.png")

In [ ]:
from IPython.display import Image as Show
from IPython.display import display

display(Show(filename=f"{W}/out/w045_9um_vs_published_2p4um.png", width=1400))

In [ ]:
print("=== SUMMARY (copy this) ===")
print("GPU:", subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
print("segment: PHerc0139 w045 at 9.362 um; checkpoints: seed42/step-075000, seed43/step-020000; both directions")
print("maps:", [os.path.basename(p) for p in maps])
print("inference seconds:", {k: round(v) for k, v in times.items()})
print(f"orientation {best}; r vs published 2.4 um map = {r_best:.3f}; shifted null = {r_null:.3f}")